# Course 6: Winning Online AI Competitions
## Session 3: Validation you can trust

**Goal:** compare KFold, GroupKFold, a time split and a single holdout on data where the choice matters, run adversarial validation, and save out-of-fold predictions in a standard format.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
def make_tabular(n=6000, seed=0):
    """A synthetic binary task with numeric columns, one high-cardinality category, groups and a date."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"id": np.arange(n)})
    for j in range(6): df[f"num{j}"] = rng.normal(size=n)
    n_shops = 300
    shop_effect = rng.normal(scale=1.0, size=n_shops)
    df["shop"] = rng.zipf(1.5, n).clip(max=n_shops) - 1                         # a few big shops, many small ones
    df["customer"] = rng.integers(0, n // 8, n)                                 # about 8 rows per customer
    cust_effect = rng.normal(scale=1.2, size=n // 8)
    df["day"] = np.sort(rng.integers(0, 365, n))
    logit = 0.8 * df.num0 - 0.6 * df.num1 + 0.5 * df.num2 * df.num3 + shop_effect[df["shop"]] + cust_effect[df.customer] + 0.004 * df.day - 1.0
    df["target"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return df
NUM = [f"num{j}" for j in range(6)]

### 1. Data with structure
Rows belong to customers (about 8 rows each), and each customer has their own tendency. The test set contains **new customers**. A validation scheme is right when it mimics that.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold, GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score
df = make_tabular(6000)
test_customers = np.random.default_rng(0).choice(df.customer.unique(), size=df.customer.nunique() // 4, replace=False)
test = df[df.customer.isin(test_customers)].reset_index(drop=True); train = df[~df.customer.isin(test_customers)].reset_index(drop=True)
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
def fit_predict(tr, te):
    """Numeric columns plus one indicator per customer id: tempting, and dangerous."""
    prep = make_column_transformer((OneHotEncoder(handle_unknown="ignore"), ["customer"]), remainder="passthrough")
    return make_pipeline(prep, LogisticRegression(C=1.0, max_iter=2000)).fit(tr[NUM + ["customer"]], tr.target).predict_proba(te[NUM + ["customer"]])[:, 1]
truth = roc_auc_score(test.target, fit_predict(train, test))
print(f"train rows {len(train)}, test rows {len(test)} | AUC on the real test set (new customers): {truth:.3f}")

### 2. Four ways to validate

In [ ]:
def cv_auc(splitter, groups=None):
    scores = [roc_auc_score(train.target.iloc[va], fit_predict(train.iloc[tr], train.iloc[va])) for tr, va in splitter.split(train, train.target, groups)]
    return np.mean(scores), np.std(scores)
res = {}
res["KFold (random rows)"] = cv_auc(KFold(5, shuffle=True, random_state=0))
res["GroupKFold (by customer)"] = cv_auc(GroupKFold(5), train.customer)
tr_, va_ = train_test_split(train, test_size=0.2, random_state=0); res["single random holdout"] = (roc_auc_score(va_.target, fit_predict(tr_, va_)), 0.0)
for name, (m, s) in res.items(): print(f"{name:28s} AUC {m:.3f} ± {s:.3f}   gap to the real test score {m - truth:+.3f}")
assert abs(res["GroupKFold (by customer)"][0] - truth) < abs(res["KFold (random rows)"][0] - truth)

Random KFold puts rows of the same customer in both the training and validation parts. The model learns each customer's tendency from their id and looks better than it will be on new customers, for whom the id carries nothing. GroupKFold keeps each customer on one side, so its score matches the test.

### 3. When time matters
If the test set comes after the training set, validate on the last part of the training period. A random split lets the model look into the future.

In [ ]:
rng = np.random.default_rng(1); n = 4000
t = np.arange(n); drift = np.cumsum(rng.normal(scale=0.05, size=n))               # a slowly drifting level
X = np.c_[t, rng.normal(size=n)]; yv = ((drift + 0.5 * X[:, 1] + rng.normal(scale=0.5, size=n)) > 0).astype(int)
tr_idx, te_idx = t < 3200, t >= 3200                                               # the real test is the future
f = lambda a, b: HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(X[a], yv[a]).predict_proba(X[b])[:, 1]
real = roc_auc_score(yv[te_idx], f(tr_idx, te_idx))
rand = np.mean([roc_auc_score(yv[va], f(tr, va)) for tr, va in KFold(5, shuffle=True, random_state=0).split(X[tr_idx])])   # positions 0..3199 are the training period
last = roc_auc_score(yv[(t >= 2560) & tr_idx], f(t < 2560, (t >= 2560) & tr_idx))
print(f"real future test AUC {real:.3f} | random KFold says {rand:.3f} | validating on the last 20% of training says {last:.3f}")
assert abs(last - real) < abs(rand - real)

### 4. Adversarial validation: is the test set different from the training set?
Label training rows 0 and test rows 1, and train a classifier to tell them apart. AUC near 0.5 means they look alike. A high AUC means they differ, and the most important features tell you where.

In [ ]:
def adversarial_auc(a, b, feats):
    both = pd.concat([a[feats], b[feats]]); lab = np.r_[np.zeros(len(a)), np.ones(len(b))]
    oof = np.zeros(len(both))
    for tr, va in KFold(5, shuffle=True, random_state=0).split(both):
        oof[va] = HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(both.iloc[tr], lab[tr]).predict_proba(both.iloc[va])[:, 1]
    return roc_auc_score(lab, oof)
print("numeric features only        : adversarial AUC %.3f" % adversarial_auc(train, test, NUM))
print("with the customer id         : adversarial AUC %.3f  <- the test customers are different people" % adversarial_auc(train, test, NUM + ["customer"]))
shifted = test.copy(); shifted["num0"] += 0.7
print("after shifting num0 in test  : adversarial AUC %.3f" % adversarial_auc(train, shifted, NUM))
assert adversarial_auc(train, shifted, NUM) > 0.6

### 5. Save out-of-fold predictions in one format, always
Every model you train should leave behind two files: OOF predictions for the training rows and predictions for the test rows. Sessions 8 and 10 depend on them.

In [ ]:
import os
def run_and_save(name, splitter, groups=None):
    oof = pd.DataFrame({"id": train.id, "fold": -1, "prediction": np.nan}); test_pred = np.zeros(len(test))
    for k, (tr, va) in enumerate(splitter.split(train, train.target, groups)):
        m = HistGradientBoostingClassifier(max_iter=150, random_state=k).fit(train.iloc[tr][NUM + ["shop"]], train.target.iloc[tr])
        oof.loc[va, "fold"] = k; oof.loc[va, "prediction"] = m.predict_proba(train.iloc[va][NUM + ["shop"]])[:, 1]
        test_pred += m.predict_proba(test[NUM + ["shop"]])[:, 1] / splitter.get_n_splits()
    os.makedirs("preds", exist_ok=True)
    oof.to_csv(f"preds/{name}_oof.csv", index=False); pd.DataFrame({"id": test.id, "prediction": test_pred}).to_csv(f"preds/{name}_test.csv", index=False)
    assert oof.prediction.notna().all() and (oof.fold >= 0).all()
    return roc_auc_score(train.target, oof.prediction)
print("OOF AUC: %.3f" % run_and_save("hgb_v1", GroupKFold(5), train.customer)); print(pd.read_csv("preds/hgb_v1_oof.csv").head(3))

Save the fold assignment once and reuse it for every model. Blending models that were validated on different folds leaks.

### TODO
1. For your competition: what does the test set contain that the training set does not (new users, later dates, other sites)? Choose the splitter that copies that.
2. Use `StratifiedGroupKFold` and compare its fold-to-fold spread with `GroupKFold`.
3. Track 'local score against leaderboard score' for 5 submissions. Do they move together?